In [1]:
import numpy as np
import networkx as nx
from scipy.ndimage import binary_dilation
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
from shapely import Polygon, distance
from skimage import measure
import networkx as nx
import skimage as ski
import dask.array as da
from shapely.strtree import STRtree
from scipy.spatial import ConvexHull, Delaunay, delaunay_plot_2d, convex_hull_plot_2d, distance
import scipy.ndimage as nd
import ast
from extract_features import subimage
from collections import defaultdict

In [ ]:
y0 = 4000
x0 = 10000
mask = da.from_array(tifffile.imread("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif"))
mask_region = mask[y0:y0+4000, x0:x0+8400]
df = pd.read_csv("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/o8p_day24_s12_results.csv", index_col="label",converters={"cells_bounds":ast.literal_eval}).drop(columns=["nucleus_percent_touching_1", "nucleus_bounds"])
# df = df[df.index.isin(np.unique(mask_region))]
cell_coords = df[["cells_i", "cells_j", "cells_bounds"]]
cell_coords = cell_coords[np.isin(cell_coords.index, np.unique(mask_region).compute())]
cell_coords

,cells_i,cells_j,cells_bounds
label,,,
40,4124.349300,14761.327527,"(4092, 14726, 4164, 14795)"
42,4214.805834,17273.754862,"(4172, 17225, 4259, 17330)"
43,4246.544919,15024.243636,"(4210, 14977, 4286, 15062)"
44,4270.996002,14948.180151,"(4232, 14897, 4305, 15007)"
45,4289.323075,16807.740878,"(4255, 16628, 4322, 16937)"
...,...,...,...
387,7490.760704,11306.700406,"(7409, 11238, 7577, 11367)"
392,7532.165284,11848.956087,"(7477, 11771, 7619, 11928)"
393,7559.780892,10970.038149,"(7520, 10920, 7605, 11021)"


In [ ]:
def prepare_box_for_contours(box, shape, pad=3):
    """Marginally pads a bounding box so that object boundaries
    are not on cropped image patch edges.
    """
    for i in range(2):
        box[i] = max(0, box[i] - pad)
        box[i+2] = min(shape[i], box[i+2] + pad)
    slices = tuple([slice(box[i], box[i+2]) for i in range(2)])
    top_left = np.array(box[:2])[None] # (1, 2)
    return slices, top_left

def make_polygons_from_mask(mask):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for rp in measure.regionprops(mask):
        # Faster to compute contours on small cell tiles than the whole image
        box_slices, box_top_left = prepare_box_for_contours(list(rp.bbox), mask.shape)
        label_mask = mask[box_slices] == rp.label

        label_cnts = np.concatenate(
            measure.find_contours(label_mask), axis=0
        )

        polygons[rp.label] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_polygons_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        polygons[lab] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_hull_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    hulls = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        hulls[lab] = ConvexHull(label_cnts + box_top_left)
    
    return hulls

def make_delaunay_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    tris = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        tris[lab] = Delaunay(label_cnts + box_top_left)
    
    return tris

def pairwise_polygon_distance(polygons_dict, dist):
    """Computes pairwise distance between all polygons in
    a dictionary. Returns a dictionary of distances.
    """
    polys = list(polygons_dict.values())
    ids = list(polygons_dict.keys())

    tree = STRtree(polys)

    id_map = dict(zip(polys, ids))

    distances = {i: {} for i in ids}

    for poly in polys:
        i = id_map[poly]

        # query nearby polygons using bounding boxes
        candidates = tree.query(poly.buffer(dist))  # 5 px search radius

        for other in candidates:
            j = id_map[other]
            if i == j:
                continue

            d = poly.distance(other)
            distances[i][j] = d
                
    return distances

def get_contour_from_label(mask, labels, df = cell_coords, scaled=True):
    contours = defaultdict(int)
    if not isinstance(labels, list):
        labels = list(labels)
    
    for label in labels:
        box = subimage(mask, df.at[label, "cells_bounds"], pad=5)

        label_cnts = np.concatenate(
        measure.find_contours(box), axis=0
            )
        
        if scaled:
            box_top_left = np.array(df.at[label, "cells_bounds"][:2])[None]
            contours[label] = label_cnts + box_top_left
        else:
            contours[label] = label_cnts
            
    return contours

In [ ]:
box40 = subimage(mask, cell_coords.at[40, "cells_bounds"], pad=5)
cont40 = measure.find_contours(box40)
tri40 = Delaunay(np.concat(cont40+np.array(cell_coords.at[40, "cells_bounds"][:2])[None], axis=0))
hull40 = ConvexHull(np.concat(cont40+np.array(cell_coords.at[40, "cells_bounds"][:2])[None], axis=0))

box42 = subimage(mask, cell_coords.at[42, "cells_bounds"], pad=5)
cont42 = measure.find_contours(box42)
tri42 = Delaunay(np.concat(cont42, axis=0))
hull42 = ConvexHull(np.concat(cont42, axis=0))

In [ ]:
newbox40 = np.zeros_like(box42)
newbox40[0:box40.shape[0],0:box40.shape[1]] = box40
both = np.concat((box42, newbox40))
both_contours = measure.find_contours(both)
both_hull = ConvexHull(np.concat(both_contours, axis=0))
both_tri = Delaunay(np.concat(both_contours, axis=0))

In [ ]:
box40 = subimage(mask, cell_coords.at[40, "cells_bounds"], pad=5)
box42 = subimage(mask, cell_coords.at[42, "cells_bounds"], pad=5)
newbox40 = np.zeros_like(box42)
newbox40[0:box40.shape[0],0:box40.shape[1]] = box40
both = np.concat((box42, newbox40))

In [ ]:
threshold = 10
both_borders = ski.segmentation.find_boundaries(both, mode="inner")*both
dilated = nd.grey_dilation(both_borders, size=(2*threshold+1, 2*threshold+1))
plt.imshow(dilated)

In [ ]:
threshold = 10
borders_region = ski.segmentation.find_boundaries(mask_region.compute())*mask_region
dilated_region = nd.grey_dilation(borders_region, size=(2*threshold+1, 2*threshold+1))
plt.imshow(dilated_region)

In [ ]:
edges = set()

# horizontal neighbors
a = dilated_region[:, :-1]
b = dilated_region[:, 1:]

mask_diff = (a != b) & (a > 0) & (b > 0)

pairs = np.stack([a[mask_diff], b[mask_diff]], axis=1)

for i, j in pairs:
    if i != j:
        edges.add(tuple(sorted((i, j))))

a = dilated_region[:-1, :]
b = dilated_region[1:, :]

mask_diff = (a != b) & (a > 0) & (b > 0)

pairs = np.stack([a[mask_diff], b[mask_diff]], axis=1)

for i, j in pairs:
    if i != j:
        edges.add(tuple(sorted((i, j))))

G = nx.Graph()
G.add_nodes_from(np.unique(mask_region.compute())[1:])
G.add_edges_from(edges)
components = list(nx.connected_components(G))

colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

cell_coords["colony_id"] = cell_coords.index.map(colony_map)


In [ ]:
max_label = dilated_region.max()

label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[dilated_region]

ids = np.unique(colony_img)
ids = ids[ids>0]
cmap = plt.get_cmap("tab20", len(ids))

label_to_color = {
    lab: cmap(i)
    for i, lab in enumerate(ids)
}

# set background / non-colony
label_to_color[-1] = (0.7, 0.7, 0.7, 1)
label_to_color[0] = (0,0,0, 1)

colors_array = np.zeros((max_label + 1, 4))

for lab, color in label_to_color.items():
    colors_array[lab] = color

rgb = colors_array[colony_img]


plt.imshow(rgb)

In [ ]:
colors_array

In [ ]:
test = np.array([0, 5, 4])
m = np.array([[0, 0, 0], [0, 1, 0], [0, 1, 2], [2, 1, 0]])

In [ ]:
test[m]

In [ ]:
G = nx.Graph()
G.add_nodes_from(np.unique(both)[1:])
G.add_edges_from(edges)
components = list(nx.connected_components(G))

In [ ]:
len(components[1])

In [ ]:
colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

cell_coords["colony_id"] = cell_coords.index.map(colony_map)

# sizes = df["colony_id_poly"].value_counts()

# valid = sizes[sizes >= 10].index

# df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
plt.imshow(rgb)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(both)
for contour in both_contours:
    ax.plot(contour[:, 1], contour[:, 0], linewidth=2)
delaunay_plot_2d(both_tri, ax = ax)
ax.axis('image')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
for contour in cont42:
    plt.plot(contour[:, 1], contour[:, 0], linewidth=2)

In [ ]:
labels = [40,42]
polys = make_polygons_from_boxes(mask, labels)
hulls = make_hull_from_boxes(mask, labels)
tris = make_delaunay_from_boxes(mask, labels)

In [ ]:
print("Min distance - polygons:",polys[labels[0]].distance(polys[labels[1]]))
print("Min distance - convex hulls:", np.min(distance.cdist(hulls[labels[0]].points, hulls[labels[1]].points)))
print("Min distance - Delaunay:", np.min(distance.cdist(tris[labels[0]].points, tris[labels[1]].points)))

In [ ]:
def process_tile(tile, threshold):

    polygons = {}
    for rp in measure.regionprops(tile):
        label = rp.label

        # bbox inside tile
        box = list(rp.bbox)
        box_slices = tuple(slice(box[i], box[i+2]) for i in range(2))

        label_mask = tile[box_slices] == label

        cnts = measure.find_contours(label_mask)
        if not cnts:
            continue

        cnt = np.concatenate(cnts, axis=0)

        # shift to global coords
        cnt[:, 0] += y1 + box[0]
        cnt[:, 1] += x1 + box[1]

        polygons[label] = Polygon(cnt)

    if not polygons:
        return []

    polys = list(polygons.values())
    ids = list(polygons.keys())

    tree = STRtree(polys)
    id_map = dict(zip(polys, ids))

    edges = []

    for poly in polys:
        i = id_map[poly]

        candidates = tree.query(poly.envelope.buffer(threshold))

        for other in candidates:
            j = id_map[other]

            if j <= i:
                continue

            d = poly.distance(other)

            if d <= threshold:
                edges.append((i, j))

    return edges

In [ ]:
polygons_dict = make_polygons_from_mask(mask)
# distances_dict = pairwise_polygon_distance(polygons_dict, 500)

In [ ]:
polygons_dict

In [ ]:
distances = pd.DataFrame(distances_dict)
distances_flat = distances.to_numpy().flatten()
distances_flat = distances_flat[~np.isnan(distances_flat)]

In [ ]:
counts, bins = np.histogram(distances_flat[distances_flat<100], bins=20)
plt.stairs(counts, bins)

In [ ]:
G = nx.Graph()

# add all cells
G.add_nodes_from(polygons_dict.keys())

threshold = 20  # adjust

for i, neighbors in distances_dict.items():
    for j, d in neighbors.items():
        if d <= threshold:
            G.add_edge(i, j)

components = list(nx.connected_components(G))

colony_map = {}
for k, comp in enumerate(components):
    for cell_id in comp:
        colony_map[cell_id] = k

df["colony_id_poly"] = df.index.map(colony_map)

sizes = df["colony_id_poly"].value_counts()

valid = sizes[sizes >= 10].index

df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
img_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day24_s12.ome.tif"
y0 = 4000
x0 = 10000

img = da.from_array(tifffile.imread(img_path))[:,y0:y0+4000, x0:x0+8400].compute()
# coverslip = da.from_zarr(tifffile.imread(coverslip_path, aszarr=True))[y0:y0+6400, x0:x0+6400]

In [ ]:
labels = df["colony_id_poly"].values
unique_labels = np.unique(labels)

# assign a color to each label
colors_map = {
    lab: plt.cm.Spectral(i / len(unique_labels))
    for i, lab in enumerate(unique_labels)
}

# override noise (-1) to black
filtered_dict = {key:(value if np.isin(key,valid)
          else (0, 0, 0, 1)) for key, value in colors_map.items()}

# build color list per point
colors = [filtered_dict[lab] for lab in labels]

fig, ax = plt.subplots(ncols=2,figsize=(20,10))

ax[0].scatter(
    df["j"],
    df["i"],
    c=colors,
    s=5
)

ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].invert_yaxis()

ax[1].imshow(ski.exposure.rescale_intensity(img[1], in_range=(0,3000)))
ax[1].set_xticks([])
ax[1].set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=2,figsize=(20,10))

for i in range(2):
    ax[i].imshow(ski.exposure.rescale_intensity(img[i], in_range=(0,3000)))
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.tight_layout()
plt.show()